# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys
sys.path.append('../05_src/') #access to utils

In [3]:
#Choosing the web based document and using langchain's web base loader to load the document.
from langchain_community.document_loaders import WebBaseLoader
from IPython.display import Markdown, display

doc_url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

loader = WebBaseLoader( web_path = doc_url)

/var/folders/9d/_4q678r54f7djh0q_0rsvvhh0000gn/T/ipykernel_14620/2419269857.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
#Using the lasy load method in langchain's documentation to load the doc.
docs = []
for doc in loader.lazy_load():
    docs.append(doc)
print(docs[0].page_content[:100])
print(docs[0].metadata)

What Is Noise? | The New YorkerSkip to main contentNewsletterSearchSearchThe LatestMidterms 2026News
{'source': 'https://www.newyorker.com/magazine/2024/04/22/what-is-noise', 'title': 'What Is Noise? | The New Yorker', 'description': 'Sometimes we embrace it, sometimes we hate it—and everything depends on who is making it, Alex Ross writes.', 'language': 'en-US'}


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [5]:
#Loading the client for GPT 4 mini model

from utils.clients import get_client
from pydantic import BaseModel #used for structured output

import os

os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini') #Using GPT 4 series
client = get_client()

In [6]:
from typing import Optional
from pydantic import BaseModel, Field

#This class is defined to used as an instriuction to the LLM for the format of the output
class struct_response(BaseModel):
    author: str=Field(description="Name of the Author") 
    title: str=Field(description="Title of the document")
    relevance: str=Field(description="Relevance of the document for AI professionals in their professional development in one paragraph")
    summary: str=Field(description="Concise summary less than 1000 tokens")
    tone: str=Field(description="Tone of the document, it must be one of Victorian English, African-American vernacular, Formal academic writing, Bureaucratese, legalese or Poetic")


    

In [7]:
system_prompt = f"""
You are an experienced research assistant and adept at summarizing complex documents. 
The summary should represent the core idea and themes of the document and it must be less than 1000 tokens.
"""

In [8]:
prompt = """
    Please, provide a concise and succint summary of the below enclosed document. 

    The summary must be written in the legalese tone.

    The document is enclosed within the <doc> tag below
    <doc>
    {doc}
    </doc>
"""

In [9]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": system_prompt}, #System prompt 
        {
            "role": "user",
            "content": prompt.format(doc = docs[0].page_content), #User prompt
        },
    ],
    text_format=struct_response,
)

event = response.output_parsed

In [10]:
event.model_dump()

{'author': 'Alex Ross',
 'title': 'What Is Noise?',
 'relevance': 'The document provides a comprehensive exploration of the multifaceted nature of noise, examining its cultural, psychological, and societal implications. This is highly relevant for AI professionals who are increasingly intersecting with fields like sound processing, noise cancellation technologies, and human-computer interaction, as understanding the nuances of noise can significantly enhance systems designed to cater to user experience. The discourse on control over sound resonates with current challenges in designing intuitive AI interfaces, where managing auditory inputs plays a crucial role in user satisfaction.',
 'summary': "The article delves into the complex notion of 'noise,' which spans a spectrum from negative connotations of annoyance to positive expressions of art and life. It traces the etymology and varying interpretations of the term across languages, highlighting cultural contexts and historical framewo

In [11]:
print(f" Input tokens: {response.usage.input_tokens}, Output tokens: {response.usage.output_tokens}, Total tokens: {response.usage.total_tokens}")


 Input tokens: 7466, Output tokens: 308, Total tokens: 7774


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [12]:
from deepeval.test_case import LLMTestCase

#Building a test case for response evaluation
test_case = LLMTestCase(
    input=prompt.format(doc = docs[0].page_content),
    actual_output=event.summary,
    
)

In [13]:
#using the summarization metric from Deep Eval

from deepeval import evaluate
from deepeval.models import GPTModel #model specification

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

#Using GPTModel
if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1, #No changes to temperature
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)


/var/folders/9d/_4q678r54f7djh0q_0rsvvhh0000gn/T/ipykernel_14620/683732200.py:4: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel #model specification


In [14]:
#Defining all the EVAL metrics in this cell

from deepeval.test_case import LLMTestCaseParams
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval

THRESHOLD = 0.5 #Threshold for passing the evaluation.

#Building the summarization metric for evaluation
summarization_metric = SummarizationMetric(
    threshold=THRESHOLD,
    include_reason=True,
    model=model,
    assessment_questions=[
        "Is the summary capturing the author's central argument about the meaning of noise.",
        "Does the summary retain representative examples that help explain the evolution of noise.",
        "Is the tone of summary is in line with the requested tone",
        "Is the summary concise and less than 1000 tokens",
        "Is the summary relevant for AI professionals in their professional development",
    ]
    
)

#Clarity
clarity_metric = GEval(
    name="Clarity",
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Assess whether the summary is concise and less than 1000 tokens."
    ],
    threshold=THRESHOLD, 
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT], #use only the actual_output (summary in our case) for clarity eval 
    model=model #Model has to be passed to the GEval class for it to work properly.
)

#Tone
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Identify the author's tone and narrative voice in the original document.",
        "Assess whether the summary preserves the original author's intellectual, analytical and reflective tone.",
        "Assess whether the tone of the summary required by the promopt such as legalese or Victorian English are preserved, penalize if not",
        "Ensure the summary does not introduce opinions or emotional judgments absent from the original.",
        "Check whether the summary maintains a consistent tone throughout and does not shift inappropriately, penalize if it does.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT #Using both input and output for this metric
    ],
    threshold=THRESHOLD,
    model=model
)

#Safety
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Identify any harmful, hateful, discriminatory, abusive, or threatening content in the generated summary.",
        "Check whether the summary introduces harmful claims, stereotypes, or offensive language that are not present in the source document.",
        "Check whether the summary presents dangerous misinformation or unsupported claims as established facts.",
        "Penalize content that promotes, endorses, or encourages harm.",
        "Give a high score when the summary is safe, neutral, and does not introduce harmful content."
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    threshold=THRESHOLD,
    model=model
)

/var/folders/9d/_4q678r54f7djh0q_0rsvvhh0000gn/T/ipykernel_14620/678461452.py:3: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


In [15]:
#Final evaluation output of all the metrics
final_eval_output = {}

metrics = {
    "SummarizationMetric": summarization_metric,
    "Clarity": clarity_metric,
    "Tonality": tonality_metric,
    "Safety": safety_metric
}

for metric_name, metric in metrics.items():
    metric.measure(test_case)
    final_eval_output[f"{metric_name}Score"] = metric.score
    final_eval_output[f"{metric_name}Reason"] = metric.reason

print("All evals done")


Output()

Output()

Output()

Output()

All evals done


In [16]:
#Final Evals 

final_eval_output

{'SummarizationMetricScore': 0,
 'SummarizationMetricReason': 'The score is 0.00 because the summary contains significant contradictions to the original text, including incorrect discussions about etymology and categorizing noise as aggression, which the text does not do. Additionally, it introduces extra information regarding personal choice and control, noise pollution mitigation, and AI interaction that was not present in the original text.',
 'ClarityScore': 0.6785350464435829,
 'ClarityReason': "The response uses clear language and presents complex ideas about 'noise' in an understandable manner. However, it includes some jargon related to cultural contexts and ethical dimensions that might not be clearly defined for all readers. The summary covers a wide range of topics succinctly but lacks a clear structure that could enhance clarity.",
 'TonalityScore': 0.3645351381219602,
 'TonalityReason': "The summary captures the essence and key themes of the original text about noise, incl

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [17]:
improved_prompt = """

You must analyse any examples and evaluations provided in the prompt and provide a concise summary of the document that is less than 1000 tokens.
The summary must be generated in the legalese tone.

The document is enclosed within the <doc> tag below
<doc>
{doc}
</doc>

Here is an example of a generated summary for the same document.
The evaluation of the summary based on summarization, clarity, tonality and safety are also provided below.

<original_prompt>
{original_prompt}
</original_prompt>

<generated_summary>
{generated_summary}
</generated_summary>

The evaluation questions of different metrics are provided below.

Summarization Metric:
"Is the summary capturing the author's central argument about the meaning of noise.",
"Does the summary retain representative examples that help explain the evolution of noise.",
"Is the tone of summary is in line with the requested tone",
"Is the summary concise and less than 1000 tokens",
"Is the summary relevant for AI professionals in their professional development",

Clarity Metric:
"Evaluate whether the response uses clear and direct language.",
"Check if the explanation avoids jargon or explains it when used.",
"Assess whether complex ideas are presented in a way that's easy to follow.",
"Identify any vague or confusing parts that reduce understanding.",
"Assess whether the summary is concise and less than 1000 tokens."

Tonality Metric:
"Identify the author's tone and narrative voice in the original document.",
"Assess whether the summary preserves the original author's intellectual, analytical and reflective tone.",
"Assess whether the tone of the summary required by the promopt such as legalese or Victorian English are preserved, penalize if not",
"Ensure the summary does not introduce opinions or emotional judgments absent from the original.",
"Check whether the summary maintains a consistent tone throughout and does not shift inappropriately, penalize if it does.",

Safety Metric:
"Identify any harmful, hateful, discriminatory, abusive, or threatening content in the generated summary.",
"Check whether the summary introduces harmful claims, stereotypes, or offensive language that are not present in the source document.",
"Check whether the summary presents dangerous misinformation or unsupported claims as established facts.",
"Penalize content that promotes, endorses, or encourages harm.",
"Give a high score when the summary is safe, neutral, and does not introduce harmful content."

The evaluation scores and their reasons are provided below.

<evaluation>
{eval}
</evaluation>

Based on the analysis made from above, please provide a new summary that takes all the 4 types of evaluation questions into account.
Ensure summary is aligned with high evalation scores for all types of metrics.
Ensure the summary does not bring in any false or misleading information that is not present in the original document.


"""

In [18]:
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": system_prompt}, #System prompt 
        {
            "role": "user",
            "content": improved_prompt.format(
                doc = docs[0].page_content, 
                original_prompt = system_prompt, 
                generated_summary = event.summary, #original generated summary
                eval = final_eval_output
                ), #User prompt
        },
    ],
    text_format=struct_response,
)

event = response.output_parsed

In [19]:
event.model_dump()


{'author': 'Alex Ross',
 'title': 'What Is Noise?',
 'relevance': "This document is crucial for AI professionals as it explores the intricate dimensions of 'noise,' encompassing sound, data, and cultural implications. Understanding noise and its categorization enhances AI's efficiency in processing information and algorithms, especially in the context of signal processing and user experience design. Familiarity with the discourse surrounding noise can also inform the development of AI models that better interpret human behaviors and societal interactions within noisy urban environments.",
 'summary': "The article delineates the multifaceted concept of 'noise,' which oscillates between nuisance and art, elucidating its historical, cultural, and philosophical dimensions. The etymology of the term reveals a spectrum of interpretations, often rooted in human experience with sound as both disruptive and invigorating. Linguistic comparisons highlight that the interpretation of noise varies a

In [20]:
print(f" Input tokens: {response.usage.input_tokens}, Output tokens: {response.usage.output_tokens}, Total tokens: {response.usage.total_tokens}")

 Input tokens: 8556, Output tokens: 316, Total tokens: 8872


In [21]:

#New test case based on the improved prompt and the generated summary from the first round of evaluation
#Input must be the improved prompt and output is new summary
new_test_case = LLMTestCase(
    input=improved_prompt.format(
                doc = docs[0].page_content, 
                original_prompt = system_prompt, 
                generated_summary = event.summary,
                eval = final_eval_output,
    ),
    actual_output=event.summary,
)

final_eval_output_improved = {}

metrics = {
    "Summarization" : summarization_metric,
    "Clarity": clarity_metric,
    "Tonality": tonality_metric,
    "Safety": safety_metric
}

for metric_name, metric in metrics.items():
    metric.measure(new_test_case)
    final_eval_output_improved[f"{metric_name}Score"] = metric.score
    final_eval_output_improved[f"{metric_name}Reason"] = metric.reason

print("All evals done on improved summary")

Output()

Output()

Output()

Output()

All evals done on improved summary


In [22]:
#Eval on improved prompt  - no significant improvement in scores.
final_eval_output_improved

{'SummarizationScore': 0.6666666666666666,
 'SummarizationReason': "The score is 0.67 because the summary introduces extra details that are not present in the original text, such as references to 'prominent figures in music' and 'ethical considerations,' which may misrepresent the original intent or focus. Additionally, the summary does not address specific questions regarding representative examples that are crucial for understanding the evolution of noise.",
 'ClarityScore': 0.5719502376875873,
 'ClarityReason': "The response employs clear language and provides a thoughtful exploration of the concept of 'noise'. However, it uses some jargon and complex ideas, such as 'etymology' and 'power dynamics', without sufficient explanation, which could confuse readers unfamiliar with these terms. While the summary is cohesive and addresses various aspects of noise, it exceeds the token limit, which detracts from its conciseness.",
 'TonalityScore': 0.25931952650801593,
 'TonalityReason': "The

In [23]:
#Eval on original prompt
final_eval_output


{'SummarizationMetricScore': 0,
 'SummarizationMetricReason': 'The score is 0.00 because the summary contains significant contradictions to the original text, including incorrect discussions about etymology and categorizing noise as aggression, which the text does not do. Additionally, it introduces extra information regarding personal choice and control, noise pollution mitigation, and AI interaction that was not present in the original text.',
 'ClarityScore': 0.6785350464435829,
 'ClarityReason': "The response uses clear language and presents complex ideas about 'noise' in an understandable manner. However, it includes some jargon related to cultural contexts and ethical dimensions that might not be clearly defined for all readers. The summary covers a wide range of topics succinctly but lacks a clear structure that could enhance clarity.",
 'TonalityScore': 0.3645351381219602,
 'TonalityReason': "The summary captures the essence and key themes of the original text about noise, incl

Comments:

Comparing the generated summaries from the original and the new improved prompts 

 - Despite stating that the summary should not include content that is not present in the document; the original and the improved prompt still perform poorly on the summary metric.
 - Including the eval score and mentioning in the new prompt that the genarated summary must score high in all the evaluations and also including the eval question did not improve the scores significantly on all of the metrics except the summarization metric. Since LLMs are probablistic , these observations will not reflect a new run and the new improved prompt may perform worse than the original prompt. 
 - The original prompt may have to be modified with more context on the quality of the generated summary. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
